In [ ]:
"""
We define a base class Run and two subclasses. In Python, we use __repr__ to provide the string representation requested (e.g., RLE[7, 9]).
"""
class Run:
    """Base interface for an encoded run."""
    def encode(self):
        raise NotImplementedError("Subclasses must implement encode()")

class RLERun(Run):
    """Stores a value and its repetition count."""
    def __init__(self, value: int, count: int):
        self.value = value
        self.count = count

    def encode(self):
        # Matches output: RLE[value, count]
        return f"RLE[{self.value}, {self.count}]"

    def __repr__(self):
        return self.encode()

class BPRun(Run):
    """Stores a list of mixed values (Bit-Packed group)."""
    def __init__(self, values: list):
        self.values = values

    def encode(self):
        # Matches output: BP[v1, v2, ...]
        return f"BP{self.values}"

    def __repr__(self):
        return self.encode()

In [ ]:
"""
Batch Encoder:
This implementation assumes we have the full list of integers at the start. It uses a greedy approach:
it looks for 8+ identical items; if found, it uses RLE. If not, it falls back to Bit-Packing.
"""
class BatchEncoder:
    def encode(self, input_list: list) -> list:
        result = []
        i = 0
        n = len(input_list)

        while i < n:
            # 1. Count consecutive identical values
            j = i
            while j < n and input_list[j] == input_list[i]:
                j += 1
            count = j - i

            # 2. Rule: If 8 or more repeats, use RLE
            if count >= 8:
                result.append(RLERun(input_list[i], count))
                i = j

            # 3. If less than 8 repeats
            else:
                remaining = n - i

                # Special Case: If we are at the very end and all items are same,
                # RLE is allowed (and preferred) even if count < 8.
                if remaining == count:
                    result.append(RLERun(input_list[i], count))
                    i = j

                # Otherwise, take the next 8 items (or whatever is left) for a BP group
                else:
                    bp_size = min(8, remaining)
                    # Slice the list to get the BP group
                    bp_values = input_list[i : i + bp_size]
                    result.append(BPRun(bp_values))
                    i += bp_size

        return result

In [ ]:
""" Decoder: The decoder simply iterates through the list of Run objects and expands them back into a flat list of integers.
"""
class Decoder:
    def decode(self, runs: list) -> list:
        output = []
        for run in runs:
            if isinstance(run, RLERun):
                # Expand the value 'count' times
                output.extend([run.value] * run.count)
            elif isinstance(run, BPRun):
                # Add all values from the stored list
                output.extend(run.values)
        return output

In [ ]:
"""Stream Encoder:
Part 4: Streaming Encoder (Follow-Up 1)
This is the core challenge. The StreamingEncoder maintains an internal buffer and a state flag (rle_active) to decide whether it is currently building a long RLE chain or waiting to fill a Bit-Packing group.
"""
class StreamingEncoder:
    def __init__(self):
        self.buffer = []        # Stores incoming numbers waiting to be processed
        self.result = []        # Stores finalized Runs

        # State variables for active RLE
        self.rle_active = False
        self.rle_value = None
        self.rle_count = 0

    def append(self, value: int):
        # Case A: We are currently inside a long RLE run
        if self.rle_active:
            if value == self.rle_value:
                self.rle_count += 1
            else:
                # The run is broken by a new value.
                # 1. Save the RLE run we just finished
                self.result.append(RLERun(self.rle_value, self.rle_count))

                # 2. Reset RLE state
                self.rle_active = False
                self.rle_value = None
                self.rle_count = 0

                # 3. Add the *new* value to the buffer to start a new group
                self.buffer.append(value)

        # Case B: We are buffering (building up to 8 items)
        else:
            self.buffer.append(value)

            # If buffer fills up to 8, we must make a decision
            if len(self.buffer) == 8:
                # Check if all 8 items are identical
                first_val = self.buffer[0]
                is_uniform = all(x == first_val for x in self.buffer)

                if is_uniform:
                    # Switch to RLE mode
                    self.rle_active = True
                    self.rle_value = first_val
                    self.rle_count = 8
                    self.buffer = [] # Clear buffer, counts are moved to state
                else:
                    # Cannot use RLE, so dump as Bit-Packing
                    self.result.append(BPRun(list(self.buffer)))
                    self.buffer = [] # Clear buffer

    def finish(self) -> list:
        # 1. If we were in the middle of an RLE run, close it.
        if self.rle_active:
            self.result.append(RLERun(self.rle_value, self.rle_count))
            self.rle_active = False

        # 2. If there are items left in the buffer (less than 8), handle them.
        if self.buffer:
            # Check if remaining items are identical (Exception rule: RLE allowed < 8 at end)
            first_val = self.buffer[0]
            is_uniform = all(x == first_val for x in self.buffer)

            if is_uniform:
                self.result.append(RLERun(first_val, len(self.buffer)))
            else:
                self.result.append(BPRun(list(self.buffer)))

            self.buffer = []

        return self.result

In [ ]:
"""Test Execution:
Here is the main block (equivalent to the Java main method) to verify all test cases mentioned in the prompt.
"""

def run_tests():
    batch_encoder = BatchEncoder()
    decoder = Decoder()

    print("--- Test 1: Short repeated sequence (last run < 8) ---")
    input1 = [1, 1, 1]
    encoded1 = batch_encoder.encode(input1)
    print("Encoded:", encoded1)
    # Expected: [RLE[1, 3]]

    print("\n--- Test 2: Mixed sequence ---")
    input2 = [1, 1, 1, 1, 2, 3, 4, 5]
    encoded2 = batch_encoder.encode(input2)
    print("Encoded:", encoded2)
    # Expected: [BP[1, 1, 1, 1, 2, 3, 4, 5]]

    print("\n--- Test 3: RLE threshold met ---")
    input3 = [1, 1, 1, 1, 1, 1, 1, 1, 2, 3, 4, 5]
    encoded3 = batch_encoder.encode(input3)
    print("Encoded:", encoded3)
    # Expected: [RLE[1, 8], BP[2, 3, 4, 5]]

    print("\n--- Test 4: Extended RLE ---")
    input4 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 3, 4, 5]
    encoded4 = batch_encoder.encode(input4)
    print("Encoded:", encoded4)
    # Expected: [RLE[1, 9], BP[2, 3, 4, 5]]

    print("\n--- Test 5: Decoder Verification (using Test 4 data) ---")
    decoded = decoder.decode(encoded4)
    print("Decoded:", decoded)
    print("Match:", decoded == input4)

    print("\n--- Test 6: Streaming Encoder ---")
    stream_encoder = StreamingEncoder()
    # Simulate streaming one by one
    for val in input4:
        stream_encoder.append(val)

    stream_encoded = stream_encoder.finish()
    print("Streaming Result:", stream_encoded)

    # Validation
    # We compare the string representations to match expectations
    expected_repr = "[RLE[1, 9], BP[2, 3, 4, 5]]"
    actual_repr = str(stream_encoded)
    print("Matches Expected:", actual_repr == expected_repr)

if __name__ == "__main__":
    run_tests()
